# Notebook 02 — Visual Validation

**Goal:** Produce an annotated MP4 that proves team assignment is working visually.

This is a sanity-check, not the final pipeline. No tracking yet.

**Output:** `outputs/day2_team_assignment_demo.mp4`

**Colour legend:**
- Red boxes = Team 0
- Blue boxes = Team 1  
- Yellow boxes = Unknown / referee

## Cell 1 — Imports

In [ ]:
import sys, io, time
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except AttributeError:
        pass

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import cv2
import numpy as np
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image as IPImage

from gaffer import config
from gaffer.detection.detector import FootballDetector
from gaffer.detection.team_assigner import TeamAssigner

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close(fig)

ROOT        = Path('..').resolve()
CLIPS_DIR   = ROOT / 'data'    / 'test_clips'
OUTPUTS_DIR = ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

CLIP        = CLIPS_DIR / 'tactical_playlist_1.mp4'
OUTPUT_PATH = OUTPUTS_DIR / 'day2_team_assignment_demo.mp4'

assert CLIP.exists(), f'Clip not found: {CLIP}'
print(f'Input  : {CLIP.name}')
print(f'Output : {OUTPUT_PATH}')
print('Imports OK')

## Cell 2 — Video Config

`tactical_playlist_1.mp4` has ~25s of pre-match intro before players appear.
We start at frame 750 (30s) so all 300 processed frames contain actual football.

In [ ]:
cap = cv2.VideoCapture(str(CLIP))
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS      = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

ACTION_START  = int(30 * FPS)   # skip pre-match intro
N_PROCESS     = 300             # frames to process
ACTION_END    = ACTION_START + N_PROCESS

print(f'Resolution   : {W}x{H}')
print(f'FPS          : {FPS}')
print(f'Total frames : {N_FRAMES}')
print(f'Processing   : frames {ACTION_START}–{ACTION_END}  '
      f'({ACTION_START/FPS:.0f}s – {ACTION_END/FPS:.0f}s)')
print(f'Output       : {OUTPUT_PATH.name}')

# Team colour palette (BGR)
TEAM_COLORS = {
     0: (0,   0,   220),   # red
     1: (220, 0,   0),     # blue
    -1: (0,   220, 220),   # yellow = unknown / referee
}

## Cell 3 — Initialize Detector & TeamAssigner

In [ ]:
detector = FootballDetector(
    conf           = config.DETECTION_CONF_THRESHOLD,
    imgsz          = config.DETECTION_IMG_SIZE,
    detect_every_n = config.DETECT_EVERY_N_FRAMES,
)
assigner = TeamAssigner()

print(f'Detector : {detector.model_path.name} ({detector.model_type})')
print(f'Conf     : {detector.conf}')
print(f'Skip     : every {detector.detect_every_n} frames')

## Cell 4 — Fit TeamAssigner on Sample Frames

We sample 8 frames evenly from the 300-frame window, run detection on each,
and use those crops to fit the K-Means model once before the main loop.

In [ ]:
def read_frame(idx):
    cap = cv2.VideoCapture(str(CLIP))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, f = cap.read()
    cap.release()
    return f if ret else None

# Evenly spaced across the 300-frame window
fit_indices   = np.linspace(ACTION_START, ACTION_END - 1, 8, dtype=int).tolist()
fit_frames    = [read_frame(i) for i in fit_indices]
fit_detections = [detector.detect_raw(f) for f in fit_frames]

crops_total = sum(
    sum(1 for d in dets if d.class_name in ('person','player','goalkeeper'))
    for dets in fit_detections
)
print(f'Fit frames   : {len(fit_frames)}')
print(f'Player crops : {crops_total}')

assigner.fit(fit_frames, fit_detections)
print(f'TeamAssigner fitted on {assigner.n_fit_samples} jersey crops')
print(f'Cluster map  : {assigner.cluster_to_team}')

for tid, bgr in assigner.team_colors_bgr.items():
    r, g, b = int(bgr[2]), int(bgr[1]), int(bgr[0])
    print(f'  Team {tid} centroid: RGB=({r},{g},{b})')

## Cell 5 — Annotate & Write Video

The main loop. Processes `N_PROCESS` frames sequentially:
1. Read frame from cap
2. `detector.detect(frame, idx)` — returns cache on non-keyframes
3. `assigner.assign(frame, dets)` — always runs (it's just a KMeans predict, ~0.1ms)
4. Draw coloured boxes + label + confidence
5. Burn timestamp into top-left corner
6. Write to VideoWriter

In [ ]:
def annotate(frame, detections):
    ann = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = det.bbox
        color = TEAM_COLORS.get(det.team_id, TEAM_COLORS[-1])

        cv2.rectangle(ann, (x1, y1), (x2, y2), color, 2)

        if det.class_name in ('ball', 'sports ball'):
            label = f'ball {det.confidence:.2f}'
        else:
            team  = f'T{det.team_id}' if det.team_id >= 0 else '?'
            label = f'{team} {det.confidence:.2f}'

        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(ann, (x1, y1 - th - 5), (x1 + tw + 3, y1), color, -1)
        cv2.putText(ann, label, (x1 + 2, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return ann

def burn_hud(frame, frame_idx, t0_count, t1_count, inference_ms):
    """Burn a small stats overlay into the top-left corner."""
    ts  = frame_idx / FPS
    lines = [
        f'Frame {frame_idx}  ({ts:.1f}s)',
        f'T0={t0_count}  T1={t1_count}',
        f'{inference_ms:.0f}ms',
    ]
    y = 20
    for line in lines:
        cv2.putText(frame, line, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(frame, line, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        y += 20
    return frame

# ── Main loop ────────────────────────────────────────────────────────────────
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(OUTPUT_PATH), fourcc, FPS, (W, H))
assert writer.isOpened(), 'VideoWriter failed to open — check codec support'

cap = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, ACTION_START)

all_lats   = []
t0_history = []
t1_history = []
annotated_samples = {}   # {local_frame_idx: annotated_frame} for display below
SAMPLE_AT = {0, 50, 100, 150, 200, 250, 299}  # frames to save for display

print(f'Processing {N_PROCESS} frames...  (every {detector.detect_every_n}rd runs inference)')
print()

for local_idx in range(N_PROCESS):
    ret, frame = cap.read()
    if not ret:
        print(f'Video ended early at local frame {local_idx}')
        break

    global_idx = ACTION_START + local_idx

    t0 = time.perf_counter()
    dets     = detector.detect(frame, global_idx)
    assigned = assigner.assign(frame, dets)
    ms = (time.perf_counter() - t0) * 1000

    t0c = sum(1 for d in assigned if d.team_id == 0)
    t1c = sum(1 for d in assigned if d.team_id == 1)
    t0_history.append(t0c)
    t1_history.append(t1c)
    all_lats.append(ms)

    ann = annotate(frame, assigned)
    ann = burn_hud(ann, global_idx, t0c, t1c, ms)
    writer.write(ann)

    if local_idx in SAMPLE_AT:
        annotated_samples[local_idx] = ann.copy()

    if (local_idx + 1) % 50 == 0 or local_idx == N_PROCESS - 1:
        elapsed = sum(all_lats) / 1000
        print(f'  [{local_idx+1:>3}/{N_PROCESS}]  '
              f'mean latency={np.mean(all_lats):.0f}ms  '
              f'T0 avg={np.mean(t0_history):.1f}  T1 avg={np.mean(t1_history):.1f}  '
              f'wall={elapsed:.1f}s')

cap.release()
writer.release()

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'\nWrote: {OUTPUT_PATH.name}  ({size_mb:.1f} MB)')

## Cell 6 — Sample Annotated Frames

Six frames pulled from the processed window. Use these to visually verify the colour
coding before opening the full video.

In [ ]:
sorted_keys = sorted(annotated_samples.keys())
n_show = min(len(sorted_keys), 6)
keys   = sorted_keys[:n_show]

fig, axes = plt.subplots(2, 3, figsize=(21, 8))
fig.suptitle(
    'Annotated Sample Frames — Red=T0  Blue=T1  Yellow=Unknown/Ref',
    fontsize=12, fontweight='bold'
)

for ax, k in zip(axes.flat, keys):
    rgb = cv2.cvtColor(annotated_samples[k], cv2.COLOR_BGR2RGB)
    ts  = (ACTION_START + k) / FPS
    t0c = t0_history[k]
    t1c = t1_history[k]
    ax.imshow(rgb)
    ax.set_title(f'frame {ACTION_START+k} ({ts:.1f}s)  T0={t0c}  T1={t1c}', fontsize=9)
    ax.axis('off')

for ax in axes.flat[n_show:]:
    ax.axis('off')

legend = [
    mpatches.Patch(color=(220/255, 0, 0),     label='Team 0'),
    mpatches.Patch(color=(0, 0, 220/255),     label='Team 1'),
    mpatches.Patch(color=(0, 220/255, 220/255), label='Unknown / Ref'),
]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 1])
show(fig)

## Cell 7 — Team Count Stability Over 300 Frames

In [ ]:
t0_arr = np.array(t0_history)
t1_arr = np.array(t1_history)
lat_arr = np.array(all_lats)
x = np.arange(len(t0_arr))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7))

ax1.plot(x, t0_arr, color='#dc3545', linewidth=1.2, label=f'Team 0 (mean={t0_arr.mean():.1f})')
ax1.plot(x, t1_arr, color='#0047ab', linewidth=1.2, label=f'Team 1 (mean={t1_arr.mean():.1f})')
ax1.fill_between(x, 0, t0_arr, alpha=0.12, color='#dc3545')
ax1.fill_between(x, 0, t1_arr, alpha=0.12, color='#0047ab')
ax1.axhline(t0_arr.mean(), color='#dc3545', linestyle='--', linewidth=0.8)
ax1.axhline(t1_arr.mean(), color='#0047ab', linestyle='--', linewidth=0.8)
ax1.set_ylabel('Players assigned')
ax1.set_title(f'Team Assignment — {N_PROCESS} frames starting at {ACTION_START/FPS:.0f}s')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

ax2.plot(x, lat_arr, color='steelblue', linewidth=0.7, alpha=0.8)
ax2.axhline(lat_arr.mean(), color='red', linestyle='--', linewidth=1.5,
            label=f'Mean {lat_arr.mean():.0f}ms (cached frames ~0ms)')
ax2.set_xlabel('Local frame #')
ax2.set_ylabel('detect + assign (ms)')
ax2.set_title('Pipeline latency — spike every 3 frames = inference run')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
show(fig)

## Cell 8 — Summary

In [ ]:
# Compute balance ratio across all frames
valid = [(t0, t1) for t0, t1 in zip(t0_arr, t1_arr) if max(t0, t1) > 0]
balance = np.mean([min(t0,t1)/max(t0,t1) for t0,t1 in valid]) if valid else 0

# Stability: frames where count shifts by > 3
t0_jumps = int(np.sum(np.abs(np.diff(t0_arr)) > 3))
t1_jumps = int(np.sum(np.abs(np.diff(t1_arr)) > 3))

print('=' * 55)
print('  VISUAL VALIDATION SUMMARY')
print('=' * 55)
print(f'\n  Output video  : {OUTPUT_PATH.name}')
print(f'  Size          : {OUTPUT_PATH.stat().st_size/1e6:.1f} MB')
print(f'  Frames written: {len(t0_arr)}')
print(f'  Duration      : {len(t0_arr)/FPS:.1f}s')
print()
print(f'  Team 0 avg detections : {t0_arr.mean():.1f}  (std {t0_arr.std():.1f})')
print(f'  Team 1 avg detections : {t1_arr.mean():.1f}  (std {t1_arr.std():.1f})')
print(f'  T0/T1 balance         : {balance:.2f}  (1.0 = equal)')
print(f'  T0 large jumps        : {t0_jumps}')
print(f'  T1 large jumps        : {t1_jumps}')
print()
print(f'  Pipeline latency')
print(f'    Mean   : {lat_arr.mean():.0f}ms  (cached frames ~0ms)')
print(f'    Median : {np.median(lat_arr):.0f}ms')
print(f'    p95    : {np.percentile(lat_arr,95):.0f}ms')
print()

if balance >= 0.6 and max(t0_jumps, t1_jumps) <= 10:
    verdict = 'PASS -- open the video and verify team colours look consistent'
elif balance >= 0.4:
    verdict = 'MARGINAL -- team assignment works but one team is under-detected'
else:
    verdict = 'FAIL -- clusters may have collapsed; try fitting on more frames'

print(f'  Verdict: {verdict}')
print()
print(f'  Open: outputs/{OUTPUT_PATH.name}')
print('=' * 55)